In [ ]:
"""
cv_lab_1.py
Каждый результат сохраняется в папку "results"
и получает читаемое имя: <input_basename>__<method>.png
Единственный вызов OpenCV: cv2.imread()
"""
import sys
import os
import numpy as np
import cv2            # только для imread
from PIL import Image, ImageDraw

: 

In [ ]:

# ---------- Утилиты ----------
def to_rgb(img_bgr):
    """Перевод BGR (cv2.imread) -> RGB, uint8"""
    return img_bgr[..., ::-1].copy()


def ensure_out_dir(out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)


def save_and_show(img_array, out_dir, base_name, method_name, show=True):
    """
    Сохранить изображение в out_dir с именем <base_name>__<method_name>.png
    и открыть его (PIL.Image.show).
    Поддерживает 2D (grayscale) и 3D (RGB).
    """
    ensure_out_dir(out_dir)

    if img_array.dtype != np.uint8:
        arr = np.clip(img_array, 0, 255).astype(np.uint8)
    else:
        arr = img_array

    safe_method = method_name.replace(" ", "_").replace("/", "_")
    filename = f"{base_name}__{safe_method}.png"
    path = os.path.join(out_dir, filename)

    pil_img = Image.fromarray(arr)
    pil_img.save(path)
    print(f"[saved] {path}")

    if show:
        try:
            pil_img.show()
        except Exception as e:
            print(f"[warning] Не удалось показать изображение: {e}")


def pad_reflect(img, pad_h, pad_w):
    """Reflect padding (mirror) для 2D или 3D массива."""
    if img.ndim == 2:
        return np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    elif img.ndim == 3:
        return np.pad(img, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='reflect')
    else:
        raise ValueError("Unsupported ndim for padding")



In [ ]:

# ---------- Фильтры ----------
def median_filter_gray(img_gray, ksize=3):
    assert ksize % 2 == 1
    h, w = img_gray.shape
    ph = ksize // 2
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+ksize, j:j+ksize].ravel()
            out[i, j] = np.median(window)
    return out


def median_filter_rgb(img_rgb, ksize=3):
    channels = []
    for c in range(3):
        channels.append(median_filter_gray(img_rgb[..., c], ksize))
    return np.stack(channels, axis=2)


def gaussian_kernel(ksize=5, sigma=1.0):
    assert ksize % 2 == 1
    ax = np.arange(-ksize//2 + 0.0, ksize//2 + 1.0)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return kernel


def convolve_gray(img_gray, kernel):
    k = kernel.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph).astype(np.float64)
    out = np.zeros((h, w), dtype=np.float64)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i, j] = np.sum(window * kernel)
    return np.clip(out, 0, 255).astype(np.uint8)


def gaussian_filter_rgb(img_rgb, ksize=5, sigma=1.0):
    kern = gaussian_kernel(ksize, sigma)
    chans = []
    for c in range(3):
        chans.append(convolve_gray(img_rgb[..., c], kern))
    return np.stack(chans, axis=2)



In [ ]:

# ---------- Морфологические операции ----------
def erosion_gray(img_gray, se=None):
    if se is None:
        se = np.ones((3,3), dtype=bool)
    k = se.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i,j] = np.min(window[se])
    return out


def dilation_gray(img_gray, se=None):
    if se is None:
        se = np.ones((3,3), dtype=bool)
    k = se.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i,j] = np.max(window[se])
    return out


def erosion_rgb(img_rgb, se=None):
    chans = []
    for c in range(3):
        chans.append(erosion_gray(img_rgb[..., c], se))
    return np.stack(chans, axis=2)


def dilation_rgb(img_rgb, se=None):
    chans = []
    for c in range(3):
        chans.append(dilation_gray(img_rgb[..., c], se))
    return np.stack(chans, axis=2)



In [ ]:

# ---------- Пороговая бинаризация ----------
def rgb_to_gray(img_rgb):
    r = img_rgb[...,0].astype(np.float32)
    g = img_rgb[...,1].astype(np.float32)
    b = img_rgb[...,2].astype(np.float32)
    y = 0.299*r + 0.587*g + 0.114*b
    return np.clip(y, 0, 255).astype(np.uint8)


def threshold_gray(img_gray, thresh=128, maxval=255, inv=False):
    if not inv:
        out = (img_gray >= thresh).astype(np.uint8) * maxval
    else:
        out = (img_gray < thresh).astype(np.uint8) * maxval
    return out


def threshold_rgb(img_rgb, thresh=128, mode='per_channel'):
    if mode == 'per_channel':
        out = np.zeros_like(img_rgb)
        for c in range(3):
            out[..., c] = threshold_gray(img_rgb[..., c], thresh)
        return out
    elif mode == 'luminance':
        gray = rgb_to_gray(img_rgb)
        th = threshold_gray(gray, thresh)
        return np.stack([th, th, th], axis=2)
    else:
        raise ValueError("Unknown mode")



In [ ]:

# ---------- Выравнивание гистограммы ----------
def equalize_hist_gray(img_gray):
    hist, _ = np.histogram(img_gray.flatten(), bins=256, range=(0,256))
    cdf = np.cumsum(hist).astype(np.float64)
    cdf_min = cdf[cdf > 0].min() if np.any(cdf > 0) else 0.0
    n = img_gray.size
    if n == cdf_min:
        lut = np.arange(256).astype(np.uint8)
    else:
        lut = np.round((cdf - cdf_min) / (n - cdf_min) * 255.0).astype(np.uint8)
    out = lut[img_gray]
    return out


def equalize_hist_rgb(img_rgb):
    arr = img_rgb.astype(np.float32)
    R = arr[...,0]; G = arr[...,1]; B = arr[...,2]
    Y  =  0.299*R + 0.587*G + 0.114*B
    Cb = 128 + (-0.168736*R - 0.331264*G + 0.5*B)
    Cr = 128 + (0.5*R - 0.418688*G - 0.081312*B)
    Y_eq = equalize_hist_gray(np.clip(Y,0,255).astype(np.uint8)).astype(np.float32)
    R2 = Y_eq + 1.402 * (Cr - 128)
    G2 = Y_eq - 0.344136 * (Cb - 128) - 0.714136 * (Cr - 128)
    B2 = Y_eq + 1.772 * (Cb - 128)
    rgb2 = np.stack([R2, G2, B2], axis=2)
    return np.clip(rgb2, 0, 255).astype(np.uint8)



In [ ]:

# ---------- Визуализация гистограммы ----------
def compute_hist_gray(img_gray):
    """
    Вычисляет гистограмму grayscale-изображения.
    Возвращает массив длины 256: hist[v] = количество пикселей со значением v.
    """
    hist, _ = np.histogram(img_gray.flatten(), bins=256, range=(0, 256))
    return hist


def render_histogram_image(hist, title="", width=512, height=300,
                            bg=(255,255,255), fg=(30,30,30), bar_color=(70,130,180)):
    """
    Превращает массив гистограммы длины 256 в RGB-изображение (NumPy uint8).

    Параметры
    ---------
    hist       : np.ndarray, shape (256,) — гистограмма яркостей
    title      : str — подпись, которая будет напечатана в верхнем левом углу
    width      : int — ширина итогового изображения в пикселях
    height     : int — высота итогового изображения в пикселях
    bg         : (R,G,B) — цвет фона
    fg         : (R,G,B) — цвет осей, делений и текста
    bar_color  : (R,G,B) — цвет столбцов гистограммы
    """
    canvas = Image.new('RGB', (width, height), bg)
    draw = ImageDraw.Draw(canvas)

    ml, mr, mt, mb = 40, 15, 30, 30      # поля: left/right/top/bottom
    plot_w = width - ml - mr
    plot_h = height - mt - mb

    max_h = int(hist.max()) if hist.max() > 0 else 1

    # --- Заголовок ---
    if title:
        draw.text((ml, 4), title, fill=fg)

    # --- Оси ---
    # вертикальная
    draw.line([(ml, mt), (ml, mt + plot_h)], fill=fg, width=1)
    # горизонтальная
    draw.line([(ml, mt + plot_h), (ml + plot_w, mt + plot_h)], fill=fg, width=1)

    # --- Деления по оси X: 0, 64, 128, 192, 255 ---
    for v in [0, 64, 128, 192, 255]:
        px = ml + int(v / 255 * (plot_w - 1))
        draw.line([(px, mt + plot_h), (px, mt + plot_h + 4)], fill=fg, width=1)
        draw.text((px - 8, mt + plot_h + 6), str(v), fill=fg)

    # --- Деления по оси Y: 0%, 50%, 100% ---
    for frac, label in [(0.0, "0"), (0.5, "50%"), (1.0, "max")]:
        py = mt + plot_h - int(frac * (plot_h - 1))
        draw.line([(ml - 4, py), (ml, py)], fill=fg, width=1)
        draw.text((1, py - 5), label, fill=fg)

    # --- Столбики / вертикальные линии ---
    for x in range(256):
        value = hist[x]
        if value == 0:
            continue
        bar_h = int((value / max_h) * (plot_h - 1))
        px = ml + int(x / 255 * (plot_w - 1))
        y0 = mt + plot_h
        y1 = y0 - bar_h
        draw.line([(px, y0), (px, y1)], fill=bar_color, width=1)

    return np.array(canvas, dtype=np.uint8)


def save_histogram_gray(img_gray, out_dir, base_name, method_name,
                         title="", show=True):
    """
    Вычислить гистограмму grayscale-изображения,
    сохранить её PNG-рисунок в out_dir с именованием по стандарту проекта.
    """
    hist = compute_hist_gray(img_gray)
    hist_img = render_histogram_image(hist, title=title)
    save_and_show(hist_img, out_dir, base_name, method_name, show=show)



In [ ]:

# ---------- Повороты на кратные 90 градусов ----------
def rotate_90_multiple(img, k=1):
    k = k % 4
    if k == 0:
        return img.copy()
    if img.ndim == 2:
        if k == 1:
            return np.flipud(np.transpose(img))
        if k == 2:
            return np.fliplr(np.flipud(img))
        if k == 3:
            return np.transpose(np.flipud(img))
    else:
        if k == 1:
            return np.flipud(np.transpose(img, (1,0,2)))
        if k == 2:
            return np.fliplr(np.flipud(img))
        if k == 3:
            return np.transpose(np.flipud(img), (1,0,2))



In [ ]:

# ---------- Демонстрация / пример использования ----------
def demo_pipeline(path, out_dir='results', show=True):
    bgr = cv2.imread(path)          # единственный вызов cv2
    if bgr is None:
        print("Не удалось прочитать изображение:", path)
        return
    rgb = to_rgb(bgr)               # BGR → RGB

    base_name = os.path.splitext(os.path.basename(path))[0]
    gray = rgb_to_gray(rgb)

    # -------- Оригинал --------
    save_and_show(rgb,  out_dir, base_name, "original_rgb",  show=show)

    # -------- Медианный фильтр --------
    med_rgb  = median_filter_rgb(rgb,  ksize=7)
    med_gray = median_filter_gray(gray, ksize=3)
    save_and_show(med_rgb,  out_dir, base_name, "median_rgb",  show=show)
    save_and_show(med_gray, out_dir, base_name, "median_gray", show=show)

    # -------- Гауссов фильтр --------
    gauss_rgb  = gaussian_filter_rgb(rgb, ksize=5, sigma=1.0)
    gauss_gray = convolve_gray(gray, gaussian_kernel(5, 1.0))
    save_and_show(gauss_rgb,  out_dir, base_name, "gaussian_rgb",  show=show)
    save_and_show(gauss_gray, out_dir, base_name, "gaussian_gray", show=show)

    # -------- Морфология --------
    se = np.array([[1,1,1],[1,1,1],[1,1,1]], dtype=bool)

    er_gray  = erosion_gray(gray, se)
    dil_gray = dilation_gray(gray, se)
    er_rgb   = erosion_rgb(rgb, se)
    dil_rgb  = dilation_rgb(rgb, se)

    save_and_show(er_gray,  out_dir, base_name, "erosion_gray",   show=show)
    save_and_show(dil_gray, out_dir, base_name, "dilation_gray",  show=show)
    save_and_show(er_rgb,   out_dir, base_name, "erosion_rgb",    show=show)
    save_and_show(dil_rgb,  out_dir, base_name, "dilation_rgb",   show=show)

    # -------- Пороговая бинаризация --------
    th_gray    = threshold_gray(gray, thresh=128)
    th_rgb_pc  = threshold_rgb(rgb,  thresh=128, mode='per_channel')
    th_rgb_lum = threshold_rgb(rgb,  thresh=128, mode='luminance')

    save_and_show(th_gray,    out_dir, base_name, "threshold_gray",              show=show)
    save_and_show(th_rgb_pc,  out_dir, base_name, "threshold_rgb_per_channel",   show=show)
    save_and_show(th_rgb_lum, out_dir, base_name, "threshold_rgb_luminance",     show=show)

    # -------- Выравнивание гистограммы --------
    # Grayscale: гистограмма ДО
    save_histogram_gray(
        gray, out_dir, base_name,
        "hist_gray_before_eq",
        title="Grayscale histogram BEFORE equalization",
        show=show
    )

    # Grayscale: эквализация и гистограмма ПОСЛЕ
    eq_gray = equalize_hist_gray(gray)
    save_and_show(eq_gray, out_dir, base_name, "equalized_gray", show=show)
    save_histogram_gray(
        eq_gray, out_dir, base_name,
        "hist_gray_after_eq",
        title="Grayscale histogram AFTER equalization",
        show=show
    )

    # RGB: эквализация через YCbCr
    eq_rgb = equalize_hist_rgb(rgb)
    save_and_show(eq_rgb, out_dir, base_name, "equalized_rgb", show=show)

    # RGB: гистограмма яркости (канал Y) ДО и ПОСЛЕ
    rgb_luma_before = rgb_to_gray(rgb)
    rgb_luma_after  = rgb_to_gray(eq_rgb)
    save_histogram_gray(
        rgb_luma_before, out_dir, base_name,
        "hist_rgb_luma_before_eq",
        title="RGB luma (Y) histogram BEFORE equalization",
        show=show
    )
    save_histogram_gray(
        rgb_luma_after, out_dir, base_name,
        "hist_rgb_luma_after_eq",
        title="RGB luma (Y) histogram AFTER equalization",
        show=show
    )

    # -------- Повороты --------
    rot90  = rotate_90_multiple(rgb, k=1)
    rot180 = rotate_90_multiple(rgb, k=2)
    rot270 = rotate_90_multiple(rgb, k=3)

    save_and_show(rot90,  out_dir, base_name, "rot90",  show=show)
    save_and_show(rot180, out_dir, base_name, "rot180", show=show)
    save_and_show(rot270, out_dir, base_name, "rot270", show=show)


if __name__ == '__main__':
    demo_pipeline("DSC01649_NOIZEE.jpg")
